# 🔄 NB-02: T5 Seq2Seq Fine-tuning (User→Response)
**Goal:** Fine-tune T5-small as a Q→A system using Claude conversations.

**Modality:** Text2Text | **Model:** google/flan-t5-small

In [ ]:
!pip install transformers datasets sentencepiece -q

In [ ]:
import json, re, random
random.seed(42)

def load_dataset(path="claude-opus-4_5-250x_jsonl.txt"):
    """Load Claude thinking dataset from JSONL."""
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def extract_fields(records):
    data = []
    for r in records:
        msgs = r["messages"]
        user = next((m["content"] for m in msgs if m["role"]=="user"), "")
        asst = next((m["content"] for m in msgs if m["role"]=="assistant"), "")
        think = re.findall(r"<think>(.*?)</think>", asst, re.DOTALL)
        response = re.sub(r"<think>.*?</think>", "", asst, flags=re.DOTALL).strip()
        data.append({
            "user": user,
            "assistant_full": asst,
            "thinking": " ".join(think).strip(),
            "response": response
        })
    return data

records = load_dataset()
data = extract_fields(records)
print(f"Loaded {len(data)} examples")
print(f"Sample user: {data[0]['user'][:80]}")
print(f"Sample think: {data[0]['thinking'][:120]}...")
print(f"Sample resp: {data[0]['response'][:120]}...")


In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
from datasets import Dataset
import torch

MODEL = "google/flan-t5-small"
tokenizer = T5Tokenizer.from_pretrained(MODEL)
model = T5ForConditionalGeneration.from_pretrained(MODEL)

def preprocess(batch):
    inputs = tokenizer(["answer: " + q for q in batch["input"]], max_length=256, truncation=True, padding="max_length")
    targets = tokenizer(batch["target"], max_length=256, truncation=True, padding="max_length")
    inputs["labels"] = [[(t if t != tokenizer.pad_token_id else -100) for t in tgt] for tgt in targets["input_ids"]]
    return inputs

ds = Dataset.from_dict({"input": [d["user"] for d in data], "target": [d["response"][:512] for d in data]})
ds = ds.map(preprocess, batched=True, remove_columns=["input","target"]).train_test_split(test_size=0.1)

collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)
args = Seq2SeqTrainingArguments(
    output_dir="./t5-claude-qa",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    evaluation_strategy="epoch",
    report_to="none"
)
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=ds["train"],
                         eval_dataset=ds["test"], data_collator=collator, tokenizer=tokenizer)
trainer.train()
print("✅ T5 fine-tuning complete!")


In [ ]:
# --- Inference ---
question = "Explain neural networks in simple terms"
inp = tokenizer("answer: " + question, return_tensors="pt").input_ids
out = model.generate(inp, max_new_tokens=150)
print(tokenizer.decode(out[0], skip_special_tokens=True))
